# RAG の回答を LLM を使わずに評価する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/22_rag/rag_eval_answer.ipynb)

NLP クックブック 22-12 ／ RAG (検索拡張生成)

RAG の回答が正しいか、資料に基づいているかを自動で確かめたい。評価のたびに LLM を呼ばずに、軽い方法で回したい。

In [1]:
import re
import unicodedata
import pandas as pd
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# 評価データ: 質問、検索された資料、システムの回答、正解 (answerable=False は「答えられない」が正解)
data = [
    {"q": "カレーはいくらですか？", "context": "看板メニューは特製ビーフカレー (950 円) です。大盛りは 100 円増しです。", "answer": "特製ビーフカレーは 950 円です。", "gold": ["950 円", "950円"], "answerable": True},
    {"q": "日曜日は何時までですか？", "context": "営業時間は平日 11 時から 15 時、土日は 11 時から 20 時です。", "answer": "日曜日は 15 時までです。", "gold": ["20 時", "20時"], "answerable": True},
    {"q": "クレジットカードは使えますか？", "context": "支払いは現金と QR コード決済に対応しています。クレジットカードは使えません。", "answer": "はい、主要なクレジットカードがご利用いただけます。", "gold": ["使えません", "使えない"], "answerable": True},
    {"q": "テイクアウトはできますか？", "context": "駐車場は店舗の裏に 3 台分あります。", "answer": "わかりません。", "gold": [], "answerable": False},
    {"q": "アレルギー対応はありますか？", "context": "看板メニューは特製ビーフカレー (950 円) です。", "answer": "はい、アレルギー対応メニューをご用意しています。", "gold": [], "answerable": False},
    {"q": "駐車場は何台分ですか？", "context": "駐車場は店舗の裏に 3 台分あります。満車の場合は近隣のコインパーキングをご利用ください。", "answer": "店舗の裏に 3 台分あり、料金は無料です。", "gold": ["3 台", "3台"], "answerable": True},
]

norm = lambda s: re.sub(r"\s", "", unicodedata.normalize("NFKC", s))
REFUSAL = re.compile(r"わかりません|分かりません|お答えできません|記載がありません|書かれていません")

nli_name = "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"
tok = AutoTokenizer.from_pretrained(nli_name)
nli = AutoModelForSequenceClassification.from_pretrained(nli_name).eval()

def faithfulness(context, answer):
    """回答の各文が資料から含意される確率の最小値 (1 文でも資料にないことを言っていれば低くなる)"""
    sents = [s for s in re.split(r"(?<=[。！？])", answer) if s.strip()]
    with torch.no_grad():
        probs = nli(**tok([context] * len(sents), sents, return_tensors="pt", padding=True, truncation=True)).logits.softmax(-1)
    return probs[:, 0].min().item()

rows = []
for d in data:
    refused = bool(REFUSAL.search(d["answer"]))
    rows.append({
        "質問": d["q"], "回答": d["answer"][:18],
        "正解を含む": any(norm(g) in norm(d["answer"]) for g in d["gold"]) if d["answerable"] else None,
        "正しく断った": refused if not d["answerable"] else None,
        "忠実性": None if refused else round(faithfulness(d["context"], d["answer"]), 2),
    })
pd.DataFrame(rows)

,質問,回答,正解を含む,正しく断った,忠実性
0,カレーはいくらですか？,特製ビーフカレーは 950 円です。,True,None,1.00
1,日曜日は何時までですか？,日曜日は 15 時までです。,False,None,0.97
2,クレジットカードは使えますか？,はい、主要なクレジットカードがご利用,False,None,0.00
3,テイクアウトはできますか？,わかりません。,None,True,NaN
4,アレルギー対応はありますか？,はい、アレルギー対応メニューをご用意,None,False,0.00
5,駐車場は何台分ですか？,店舗の裏に 3 台分あり、料金は無料,True,None,0.01


In [2]:
df = pd.DataFrame(rows)
print(f"正解を含む割合 (答えられる質問): {df['正解を含む'].dropna().mean():.2f}")
print(f"正しく断った割合 (答えられない質問): {df['正しく断った'].dropna().mean():.2f}")
print(f"忠実性が 0.5 未満の回答 (資料にないことを述べている疑い): {(df['忠実性'].dropna() < 0.5).sum()} 件")

正解を含む割合 (答えられる質問): 0.50
正しく断った割合 (答えられない質問): 0.50
忠実性が 0.5 未満の回答 (資料にないことを述べている疑い): 3 件


- 3 つの観点を分けて見る: 正しさ (正解を含むか)、忠実性 (資料に基づいているか)、棄却 (答えがないときに断れるか)。正しいが資料にない、資料どおりだが質問に答えていない、ということが起こる。
- 最後の例は「3 台分」は正しいが「料金は無料」は資料にない。正解を含むかだけを見ていると、この種の付け足し (ハルシネーション) を見逃す。
- NLI は数値の取り違えに弱い。2 つ目の例 (資料は「土日は 20 時まで」、回答は「15 時まで」) は誤りなのに忠実性が高く出ている。正解との照合と必ず併用する。
- NLI による忠実性は LLM による判定より粗いが、速くて安く、結果が安定している。変更のたびに回す回帰テストに向く。
- 最終的な品質の確認には、人手の評価か、LLM による評価 (この章の「RAG の回答の正しさを LLM に判定させる」、評価の章の LLM-as-a-judge) を併用する。